# DocMind · 02 — Fine-tune the answer model (Qwen2.5-7B-Instruct, QLoRA)

Fine-tunes **Qwen2.5-7B-Instruct** into DocMind's answer model and exports it as a **GGUF** file that
Ollama serves locally. The model is trained on DocMind's *exact* prompt format — the same system prompt,
`[CHUNK n | source: FILE | page P]` passages and `Sources:` section that `generation.py` sends — so what it
learns transfers directly.

**Runtime:** `Runtime → Change runtime type → T4 GPU`. About 2–2.5 hours: ~10 min setup, ~90 min training,
~30 min GGUF export. A T4 trains a 7B model at only ~340 tokens/s, which is what `SCALE` trades against.
Checkpoints go to Google Drive, so a disconnected session can resume.

**Output (Google Drive → `MyDrive/docmind/`, suffixed with `RUN_NAME`):**
- `docmind-qwen2.5-7b-<RUN_NAME>.Q4_K_M.gguf` (~4.7 GB) — download into the repo's `models/` folder and
  register it with `ollama create` (see `training/Modelfile`)
- `lora_adapter_<RUN_NAME>/` — the LoRA weights alone (~160 MB), a lightweight backup

### What the training data teaches
| Behaviour | Source | Target answer |
|---|---|---|
| Answer from the right passage, completely | SQuAD 2.0 answerable, answering paragraph shuffled among same-article paragraphs | the **passage sentence that contains the answer** |
| Combine passages | HotpotQA *distractor* (non yes/no), 2 supporting paragraphs plus 2–3 distractors | the supporting sentence that contains the answer |
| Read tables in PDF-extracted text | synthetic tables (rows of space-separated values under a header) | "According to Table N, the X of Y is Z." |
| Refuse when the passages are on-topic but don't answer | SQuAD 2.0 unanswerable; HotpotQA with its supporting paragraphs removed; table questions about rows/columns that aren't there | the canonical refusal |
| Fluent answers, extraction and summaries | Dolly-15k `closed_qa`, `information_extraction`, `summarization` (with their passages) | Dolly's written response |

**Why v2.** v1 used SQuAD/HotpotQA's bare answer spans ("32", "no") as targets, and the model learned to
answer in fragments — dropping the second half of two-part questions and losing to the untuned model on the
gold set (88.5% vs 90.4%). It also invented numbers for table questions it should have refused; v1 had no
tables in its training data. v2 targets complete, grounded sentences, adds tables, and uses a lower learning
rate so the base model's behaviour is adjusted rather than overwritten.

**Result of v2 (gold set, 261 questions):** 84.7% — below v1 (88.5–90.0%) and the untuned base model (90.4%).
Trained to answer with a passage sentence, it learned to copy *a relevant-looking* sentence, often not the one
containing the answer. DocMind therefore keeps an untuned model (llama3.1, 93.5%). A next attempt should use
teacher-written answers (a natural sentence built around the gold span) rather than raw spans or raw sentences.

**None of DocMind's gold-set documents are used** — they stay test-only, so the local gold-set score
measures generalisation, not memorisation.

In [ ]:
%%capture
!pip install -q unsloth

In [ ]:
import json, os, random, re, collections

RUN_NAME        = "v2"     # output names and checkpoint folder; change it for a fresh run
SEED            = 3407
BASE_MODEL      = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"   # 4-bit copy of Qwen/Qwen2.5-7B-Instruct (Apache-2.0)
MAX_SEQ_LENGTH  = 2048
LEARNING_RATE   = 1e-4     # v1 used 2e-4 and over-wrote the base model's answer style
# Training mix (examples) at SCALE=1.0.  A T4 trains this model at ~340 tokens/s: SCALE=0.3
# (~2,100 examples, ~900 tokens each) takes ~1.5 h.
SCALE = 0.3
MIX = {
    "squad_answerable":   1500,
    "squad_unanswerable":  600,
    "hotpot_answerable":  1200,
    "hotpot_missing":      350,
    "table_answerable":    750,
    "table_unanswerable":  450,
    "dolly_grounded":     1300,
    "dolly_summary":       500,
    "dolly_mismatch":      200,
}
MIX = {k: int(v * SCALE) for k, v in MIX.items()}
REFUSAL_KINDS   = {"squad_unanswerable", "hotpot_missing", "table_unanswerable", "dolly_mismatch"}
DRIVE_DIR       = "/content/drive/MyDrive/docmind"
RUN_QUICK_EVAL  = False    # True adds ~25 min: base vs fine-tuned on held-out SQuAD/HotpotQA questions

random.seed(SEED)
print(MIX, "total:", sum(MIX.values()))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

## 1 · DocMind's prompt format

Copied from `generation.py` (`SYSTEM_PROMPT` and `_build_user_prompt`). **Keep these in sync** — if the app's
prompt changes, re-run this notebook.

In [ ]:
SYSTEM_PROMPT = """\
You are a document lookup assistant. Your ONLY job is to answer questions \
using the text passages provided below — nothing else.

Rules you must follow without exception:
- Base your answer EXCLUSIVELY on the passages marked [CHUNK n | source: FILE].
- After your answer, list every source file you drew on under the heading \
"Sources:" — one file name per line, no duplicates.
- If the provided passages do not contain enough information to answer the \
question, respond with exactly: "I don't have enough information in the \
provided documents to answer this question." Do NOT speculate or use outside \
knowledge.
- Do not make up file names, page numbers, or details that are not in the \
passages.
- Keep your answer concise and factual.\
"""

REFUSAL = "I don't have enough information in the provided documents to answer this question."

def build_user_prompt(chunks, question):
    """chunks: list of (source_file, page, text) — same layout as generation._build_user_prompt."""
    blocks = []
    for i, (source, page, text) in enumerate(chunks, start=1):
        page_tag = f" | page {page}" if page else ""
        blocks.append(f"[CHUNK {i} | source: {source}{page_tag}]\n{text.strip()}")
    context = "\n\n---\n\n".join(blocks)
    return (f"PASSAGES:\n\n{context}\n\n---\n\nQUESTION: {question}\n\n"
            "Answer using only the passages above. List sources at the end.")

def answer_with_sources(answer, sources):
    return answer.strip() + "\n\nSources:\n" + "\n".join(dict.fromkeys(sources))

def as_file(title):
    return re.sub(r"[^\w\- ]+", "", title).strip().replace(" ", "_")[:60] + ".pdf"

def example(chunks, question, target):
    return {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(chunks, question)},
        {"role": "assistant", "content": target},
    ]}

_SENTENCE_END = re.compile(r"[.!?](?=\s|$)")

def sentence_containing(text, start, length):
    """The full sentence of `text` around characters [start, start+length), or None if it's unusable."""
    begin = 0
    for m in _SENTENCE_END.finditer(text, 0, start):
        begin = m.end()
    m = _SENTENCE_END.search(text, start + length)
    end = m.end() if m else len(text)
    sentence = text[begin:end].strip()
    return sentence if 4 <= len(sentence.split()) <= 60 else None

## 2 · Build the training mix

In [ ]:
from datasets import load_dataset

squad  = load_dataset("rajpurkar/squad_v2")
hotpot = load_dataset("hotpotqa/hotpot_qa", "distractor")
dolly  = load_dataset("databricks/databricks-dolly-15k")["train"]

def squad_examples(split, n_answerable, n_unanswerable):
    by_title = collections.defaultdict(list)
    for ex in split:
        by_title[ex["title"]].append(ex)
    rows = [ex for exs in by_title.values() for ex in exs]
    random.shuffle(rows)
    out, need = [], {"a": n_answerable, "u": n_unanswerable}
    for ex in rows:
        kind = "a" if ex["answers"]["text"] else "u"
        if need[kind] == 0:
            continue
        if kind == "a":
            span, start = ex["answers"]["text"][0], ex["answers"]["answer_start"][0]
            answer = sentence_containing(ex["context"], start, len(span))
            if answer is None:
                continue
        contexts = list(dict.fromkeys(e["context"] for e in by_title[ex["title"]]))
        others = [c for c in contexts if c != ex["context"]]
        passages = [ex["context"]] + random.sample(others, min(len(others), random.randint(2, 4)))
        random.shuffle(passages)
        source = as_file(ex["title"])
        pages = random.sample(range(1, 40), len(passages))
        chunks = [(source, p, c) for p, c in zip(pages, passages)]
        target = answer_with_sources(answer, [source]) if kind == "a" else REFUSAL
        out.append(example(chunks, ex["question"], target) | {"kind": f"squad_{'answerable' if kind == 'a' else 'unanswerable'}"})
        need[kind] -= 1
        if not any(need.values()):
            break
    return out

def hotpot_examples(split, n_answerable, n_missing):
    rows = list(range(len(split)))
    random.shuffle(rows)
    out, made_a, made_m = [], 0, 0
    for idx in rows:
        if made_a >= n_answerable and made_m >= n_missing:
            break
        ex = split[idx]
        titles, sentences = ex["context"]["title"], ex["context"]["sentences"]
        gold = set(ex["supporting_facts"]["title"])
        paragraphs = [(as_file(t), random.randint(1, 30), " ".join(s).strip()) for t, s in zip(titles, sentences)]
        supporting = [p for p, t in zip(paragraphs, titles) if t in gold]
        distractors = [p for p, t in zip(paragraphs, titles) if t not in gold]
        if made_a < n_answerable:
            if ex["answer"].lower() in ("yes", "no"):
                continue   # a bare yes/no target is exactly the fragment style v2 avoids
            # the supporting sentence that states the answer becomes the target
            facts = [sentences[titles.index(t)][i] for t, i in zip(ex["supporting_facts"]["title"],
                                                                     ex["supporting_facts"]["sent_id"])
                     if t in titles and i < len(sentences[titles.index(t)])]
            stating = [s.strip() for s in facts if ex["answer"].lower() in s.lower() and 4 <= len(s.split()) <= 60]
            if not stating:
                continue
            chunks = supporting + random.sample(distractors, min(len(distractors), random.randint(2, 3)))
            random.shuffle(chunks)
            gold_files = [as_file(t) for t in titles if t in gold]
            out.append(example(chunks, ex["question"], answer_with_sources(stating[0], gold_files)) | {"kind": "hotpot_answerable"})
            made_a += 1
        else:
            kept = random.sample(distractors, min(len(distractors), 4))
            out.append(example(kept, ex["question"], REFUSAL) | {"kind": "hotpot_missing"})
            made_m += 1
    return out

TABLE_TOPICS = [
    ("model", ["Accuracy", "F1", "Precision", "Recall", "Latency (ms)", "Params (M)"],
     ["ResNet50", "DenseNet121", "ViT-B/16", "EfficientNet-B0", "MobileNetV3", "ConvNeXt-T", "Swin-T", "BiLSTM"]),
    ("region", ["Revenue ($M)", "Growth (%)", "Customers", "Churn (%)", "Margin (%)"],
     ["North", "South", "East", "West", "Central", "Pacific", "Atlantic"]),
    ("dataset", ["Train", "Validation", "Test", "Classes", "Avg. length"],
     ["CIFAR-10", "ImageNet-1k", "SQuAD", "COCO", "MNIST", "IMDB", "AG News"]),
    ("component", ["Weight (kg)", "Cost ($)", "Lifetime (h)", "Failure rate (%)"],
     ["Motor", "Gearbox", "Sensor", "Controller", "Battery", "Housing"]),
]

def random_value(column):
    if "%" in column or column in ("Accuracy", "F1", "Precision", "Recall"):
        return f"{random.uniform(40, 99):.1f}"
    if column in ("Customers", "Train", "Validation", "Test"):
        return f"{random.randint(1_000, 900_000):,}"
    return f"{random.uniform(1, 500):.2f}"

def table_examples(n_answerable, n_unanswerable):
    out = []
    for i in range(n_answerable + n_unanswerable):
        kind = "table_answerable" if i < n_answerable else "table_unanswerable"
        entity, all_cols, all_rows = random.choice(TABLE_TOPICS)
        cols = random.sample(all_cols, random.randint(3, len(all_cols)))
        rows = random.sample(all_rows, random.randint(3, min(6, len(all_rows) - 1)))
        values = {(r, c): random_value(c) for r in rows for c in cols}
        n = random.randint(1, 12)
        # PDF extraction flattens a table to a header line and space-separated rows
        table = f"Table {n}: Comparison of each {entity}.\n{entity.capitalize()} " + " ".join(cols) + "\n" + \
                "\n".join(f"{r} " + " ".join(values[(r, c)] for c in cols) for r in rows)
        source = f"report_{random.randint(1000, 9999)}.pdf"
        chunks = [(source, random.randint(2, 30), table)]
        chunks += [(f"reference_{j:04d}.pdf", 1, d["context"]) for j, d in enumerate(random.sample(dolly_context_rows, 1))]
        random.shuffle(chunks)
        if kind == "table_answerable":
            r, c = random.choice(rows), random.choice(cols)
            question = f"What is the {c} of {r}{' in Table ' + str(n) if random.random() < 0.5 else ''}?"
            target = answer_with_sources(f"According to Table {n}, the {c} of {r} is {values[(r, c)]}.", [source])
        else:
            if random.random() < 0.5:   # a row that is not in the table
                r, c = random.choice([x for x in all_rows if x not in rows]), random.choice(cols)
            else:                       # a column that is not in the table
                missing = [x for x in all_cols if x not in cols]
                if not missing:
                    continue
                r, c = random.choice(rows), random.choice(missing)
            question = f"What is the {c} of {r}?"
            target = REFUSAL
        out.append(example(chunks, question, target) | {"kind": kind})
    return out

def split_passage(text, max_words=180):
    # Long Dolly contexts become several chunks, like a DocMind document would.
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks, current = [], []
    for s in sentences:
        if current and len(" ".join(current + [s]).split()) > max_words:
            chunks.append(" ".join(current)); current = []
        current.append(s)
    if current:
        chunks.append(" ".join(current))
    return chunks

dolly_context_rows = [r for r in dolly if r["context"].strip() and r["response"].strip()]

def dolly_examples(n_grounded, n_summary, n_mismatch):
    grounded = [r for r in dolly_context_rows if r["category"] in ("closed_qa", "information_extraction")]
    summaries = [r for r in dolly_context_rows if r["category"] == "summarization"]
    random.shuffle(grounded); random.shuffle(summaries)
    out = []
    for i, r in enumerate(grounded[:n_grounded]):
        source = f"document_{i:04d}.pdf"
        own = [(source, p + 1, c) for p, c in enumerate(split_passage(r["context"]))]
        distractors = [(f"reference_{j:04d}.pdf", 1, d["context"]) for j, d in
                       enumerate(random.sample(dolly_context_rows, 1))]
        chunks = own + distractors
        random.shuffle(chunks)
        out.append(example(chunks, r["instruction"], answer_with_sources(r["response"], [source])) | {"kind": "dolly_grounded"})
    for i, r in enumerate(summaries[:n_summary]):
        source = f"report_{i:04d}.pdf"
        chunks = [(source, p + 1, c) for p, c in enumerate(split_passage(r["context"]))]
        out.append(example(chunks, r["instruction"], answer_with_sources(r["response"], [source])) | {"kind": "dolly_summary"})
    unused = grounded[n_grounded:]
    for i, r in enumerate(unused[:n_mismatch]):
        # The question's own passage is withheld: only unrelated passages are shown.
        chunks = [(f"reference_{j:04d}.pdf", 1, d["context"]) for j, d in
                  enumerate(random.sample([x for x in dolly_context_rows if x is not r], 3))]
        out.append(example(chunks, r["instruction"], REFUSAL) | {"kind": "dolly_mismatch"})
    return out

train_rows = (
    squad_examples(squad["train"], MIX["squad_answerable"], MIX["squad_unanswerable"])
    + hotpot_examples(hotpot["train"], MIX["hotpot_answerable"], MIX["hotpot_missing"])
    + table_examples(MIX["table_answerable"], MIX["table_unanswerable"])
    + dolly_examples(MIX["dolly_grounded"], MIX["dolly_summary"], MIX["dolly_mismatch"])
)
random.shuffle(train_rows)
kinds = collections.Counter(r["kind"] for r in train_rows)
refusals = sum(v for k, v in kinds.items() if k in REFUSAL_KINDS)
print(kinds, f"\nrefusal share: {refusals / len(train_rows):.1%}")
for kind in ("squad_answerable", "hotpot_answerable", "table_answerable"):
    sample = next(r for r in train_rows if r["kind"] == kind)
    print(f"\n--- {kind} ---\nQ:", sample["messages"][1]["content"].split("QUESTION: ")[1].split("\n")[0],
          "\n>>>", sample["messages"][2]["content"])

## 3 · Load Qwen2.5-7B-Instruct in 4-bit and attach LoRA adapters

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

In [ ]:
from datasets import Dataset

def render(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False)}

train_ds = Dataset.from_list([{"messages": r["messages"]} for r in train_rows]).map(render, remove_columns=["messages"])
lengths = [len(tokenizer(t).input_ids) for t in train_ds["text"][:500]]
print(f"examples: {len(train_ds):,}  token length (sample of 500): mean {sum(lengths) / len(lengths):.0f}, "
      f"max {max(lengths)}, over {MAX_SEQ_LENGTH}: {sum(l > MAX_SEQ_LENGTH for l in lengths)}")

## 4 · Optional held-out check (before training)

Off by default (`RUN_QUICK_EVAL`): DocMind's own gold set, run locally after export, is the real test.

In [ ]:
import torch

def normalize(s):
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(re.sub(r"[^a-z0-9. ]", " ", s).split())

def held_out(n_squad_a=60, n_squad_u=30, n_hotpot=60):
    random.seed(SEED + 1)
    rows = (squad_examples(squad["validation"], n_squad_a, n_squad_u)
            + hotpot_examples(hotpot["validation"], n_hotpot, 0))
    random.seed(SEED)
    return rows

def generate(messages, max_new_tokens=160):
    prompt = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def fits(r, margin=200):
    prompt = tokenizer.apply_chat_template(r["messages"][:2], tokenize=False, add_generation_prompt=True)
    return len(tokenizer(prompt).input_ids) <= MAX_SEQ_LENGTH - margin

def quick_eval(rows, label):
    FastLanguageModel.for_inference(model)
    stats = collections.Counter()
    for r in [r for r in rows if fits(r)]:   # Unsloth refuses inputs longer than MAX_SEQ_LENGTH
        target = r["messages"][2]["content"]
        pred = generate(r["messages"])
        refused = "enough information" in pred.lower()
        if target == REFUSAL:
            stats["refusal_total"] += 1
            stats["refusal_ok"] += refused
        else:
            gold = normalize(target.split("\n\nSources:")[0]).rstrip(".")
            stats["answer_total"] += 1
            stats["answer_ok"] += (not refused) and gold in normalize(pred)
            stats["false_refusal"] += refused
    FastLanguageModel.for_training(model)
    print(f"[{label}] answer contains gold: {stats['answer_ok']}/{stats['answer_total']}  "
          f"false refusals: {stats['false_refusal']}  correct refusals: {stats['refusal_ok']}/{stats['refusal_total']}")
    return dict(stats)

eval_rows = held_out() if RUN_QUICK_EVAL else []
base_stats = quick_eval(eval_rows, "base") if RUN_QUICK_EVAL else None

## 5 · Train (loss on the assistant's answer only)

Checkpoints go to `checkpoints-<RUN_NAME>` on Drive every 100 steps. If the session drops, re-run the
cells above and then this cell — it resumes from the last checkpoint of this run.

In [ ]:
import glob
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

checkpoint_dir = f"{DRIVE_DIR}/checkpoints-{RUN_NAME}"
config_kwargs = dict(
    dataset_text_field="text",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    weight_decay=0.01,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    output_dir=checkpoint_dir,
    seed=SEED,
    report_to="none",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
)
try:   # TRL renamed max_seq_length -> max_length; accept either version
    args = SFTConfig(max_seq_length=MAX_SEQ_LENGTH, **config_kwargs)
except TypeError:
    args = SFTConfig(max_length=MAX_SEQ_LENGTH, **config_kwargs)

try:   # TRL 1.x calls the tokenizer argument processing_class
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=train_ds, args=args)
except TypeError:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, args=args)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
resume = bool(glob.glob(f"{checkpoint_dir}/checkpoint-*"))
print("resuming from checkpoint" if resume else "starting fresh")
stats = trainer.train(resume_from_checkpoint=resume)
print(stats)

In [ ]:
tuned_stats = quick_eval(eval_rows, "fine-tuned") if RUN_QUICK_EVAL else None
model.save_pretrained(f"{DRIVE_DIR}/lora_adapter_{RUN_NAME}")
tokenizer.save_pretrained(f"{DRIVE_DIR}/lora_adapter_{RUN_NAME}")
with open(f"{DRIVE_DIR}/training_summary_{RUN_NAME}.json", "w") as f:
    json.dump({"run": RUN_NAME, "base_model": BASE_MODEL, "learning_rate": LEARNING_RATE, "mix": MIX,
               "examples": len(train_ds), "quick_eval": {"base": base_stats, "finetuned": tuned_stats}}, f, indent=2)

## 6 · Export to GGUF (Q4_K_M) for Ollama

Unsloth merges the LoRA weights into the base model and quantises with llama.cpp. Q4_K_M (~4.7 GB) fits an
8 GB GPU with room for an 8K context.

In [ ]:
import shutil
from google.colab import drive as gdrive

export_dir = "/content/gguf"
model.save_pretrained_gguf(export_dir, tokenizer, quantization_method="q4_k_m")
# Unsloth may write to a sibling folder (e.g. /content/gguf_gguf/), so search /content -- but not the
# mounted Drive, which holds GGUFs from earlier runs that could be picked up instead of this one.
produced = sorted((p for p in glob.glob("/content/**/*.gguf", recursive=True) if not p.startswith("/content/drive/")),
                  key=os.path.getmtime)
print(produced)
gguf = [p for p in produced if "q4_k_m" in p.lower()][-1]   # newest Q4_K_M export
target = f"{DRIVE_DIR}/docmind-qwen2.5-7b-{RUN_NAME}.Q4_K_M.gguf"
shutil.copy(gguf, target)
print(f"saved {target} ({os.path.getsize(target) / 1e9:.2f} GB)")
gdrive.flush_and_unmount()   # waits until the upload to Drive has finished
print("Drive sync complete — download the file into the repo's models/ folder")